# 05 — Split + Time Series CV (quarter-end expanding-window folds)

**Purpose.** Builds the chronological Train/Embargo/Test split and the
expanding-window cross-validation folds used for model training and
evaluation, and persists the fold boundaries so they stay fixed across every
notebook that uses them.

**Why quarter-end grain.** The model's target is a quarter-over-quarter ZHVI
decline, a quarterly concept. Cross-validation folds are built directly on
quarter-end months (March, June, September, December) rather than on raw
monthly rows, so a fold boundary always lands on a real calendar quarter
boundary instead of splitting partway through one.

**Why persist to JSON.** Zillow's source file grows forward every time it's
re-downloaded, so the number of rows in `county_month_check` isn't fixed over
time. Recomputing a split from scratch each run would let fold boundaries
drift silently as new months arrive. Instead, folds are computed once here and
written to `../outputs/cv_folds.json`; every later notebook should load that
file rather than regenerating its own split, so all downstream training and
evaluation use identical folds.

Cutoffs (from `docs/phase0_framing.md`'s Train/Test Split section):
- Train: through 2021-09-30
- Embargo: 2021-10-01 to 2021-12-31 (excluded from both Train and Test)
- Test: 2022-01-01 through 2026-03-31

Sections 3 through 5 build the row-level Train/Embargo/Test split and a
leakage check, and save `train_monthly.csv` / `embargo_monthly.csv` /
`test_monthly.csv` for later phases. Sections 6 through 8 build the
quarter-end expanding-window CV folds and persist them.


## 1. Setup

In [13]:
import os
import json
import pandas as pd
from sqlalchemy import text

from housing_risk.config import get_engine

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 20)


In [14]:
# --- Database connection ---
# Shared with every notebook in this project via src/housing_risk/config.py,
# so credentials and cutoff dates only have to be defined once.
engine = get_engine()

with engine.connect() as conn:
    print(conn.execute(text("SELECT version();")).scalar())


PostgreSQL 15.18 (Homebrew) on aarch64-apple-darwin25.4.0, compiled by Apple clang version 21.0.0 (clang-2100.0.123.102), 64-bit


In [15]:
# Split and CV constants, shared across the project via src/housing_risk/config.py
# (this used to be defined inline here; see that module for the reasoning
# behind the specific cutoff dates).
from housing_risk.config import (
    TRAIN_END,
    EMBARGO_START,
    EMBARGO_END,
    TEST_START,
    TEST_END,
    QUARTER_END_MONTHS,
    CV_INITIAL_TRAIN_QUARTERS,
    CV_FOLD_VAL_QUARTERS,
)

FOLDS_PATH = "../outputs/cv_folds.json"


## 2. Load `county_month_check` and confirm real months

`month` comes back as a real `Timestamp` from Postgres (no `Period`/CSV
round-trip), so comparisons against the cutoffs above are exact.


In [16]:
with engine.connect() as conn:
    has_table = conn.execute(
        text("SELECT to_regclass(\'county_month_check\') IS NOT NULL")
    ).scalar()
assert has_table, (
    "county_month_check doesn't exist yet -- run 03_join_sql_load.ipynb first "
    "(it builds and verifies this table)."
)

joined = pd.read_sql("SELECT county_fips, month, zhvi, mortgage_rate FROM county_month_check", engine)
joined["month"] = pd.to_datetime(joined["month"])

months = sorted(joined["month"].unique())
print(f"{len(months)} distinct months: {months[0].date()} to {months[-1].date()}")

# Stop and report if any cutoff isn't a real month present in the data, rather
# than silently trusting a hardcoded date.
for label, value in [("TRAIN_END", TRAIN_END), ("EMBARGO_END", EMBARGO_END), ("TEST_END", TEST_END)]:
    assert value in months, f"STOP AND REPORT: {label}={value.date()} is not a real month in county_month_check"
print("TRAIN_END, EMBARGO_END, TEST_END all confirmed present as real months.")


320 distinct months: 2000-01-31 to 2026-08-31
TRAIN_END, EMBARGO_END, TEST_END all confirmed present as real months.


## 3. Apply the Three Date Cutoffs

Splits the loaded rows into Train, Embargo, and Test by month. This is a
row-level split, independent of the CV fold construction in sections 6-8
below, and feeds `train_monthly.csv` / `embargo_monthly.csv` /
`test_monthly.csv` for later phases.


In [17]:
train_df   = joined[joined["month"] <= TRAIN_END].copy()
embargo_df = joined[(joined["month"] >= EMBARGO_START) & (joined["month"] <= EMBARGO_END)].copy()
test_df    = joined[(joined["month"] >= TEST_START) & (joined["month"] <= TEST_END)].copy()

assigned_mask = (
    (joined["month"] <= TRAIN_END)
    | ((joined["month"] >= EMBARGO_START) & (joined["month"] <= EMBARGO_END))
    | ((joined["month"] >= TEST_START) & (joined["month"] <= TEST_END))
)
unassigned_df = joined[~assigned_mask]

print(f"Train rows:      {len(train_df):,}")
print(f"Embargo rows:    {len(embargo_df):,}")
print(f"Test rows:       {len(test_df):,}")
print(f"Unassigned rows: {len(unassigned_df):,}  (outside all three windows, e.g. data past 2026-03-31)")


Train rows:      529,214
Embargo rows:    9,063
Test rows:       156,364
Unassigned rows: 15,355  (outside all three windows, e.g. data past 2026-03-31)


## 4. Leakage Check

No county-month should appear in more than one of the three windows.


In [18]:
train_keys   = set(zip(train_df["county_fips"], train_df["month"]))
embargo_keys = set(zip(embargo_df["county_fips"], embargo_df["month"]))
test_keys    = set(zip(test_df["county_fips"], test_df["month"]))

assert len(train_keys & embargo_keys) == 0, "Leakage: Train/Embargo overlap found!"
assert len(embargo_keys & test_keys) == 0, "Leakage: Embargo/Test overlap found!"
assert len(train_keys & test_keys) == 0, "Leakage: Train/Test overlap found!"
print("No overlap between Train/Embargo/Test windows.")


No overlap between Train/Embargo/Test windows.


## 5. Save row-level splits

Saves the three splits for later phases to build on.


In [19]:
os.makedirs("../data/processed", exist_ok=True)

train_df.to_csv("../data/processed/train_monthly.csv", index=False)
embargo_df.to_csv("../data/processed/embargo_monthly.csv", index=False)
test_df.to_csv("../data/processed/test_monthly.csv", index=False)
print("Saved train_monthly.csv, embargo_monthly.csv, test_monthly.csv to data/processed/")


Saved train_monthly.csv, embargo_monthly.csv, test_monthly.csv to data/processed/


## 6. Quarter-ends by window

Quarter-end months (Mar/Jun/Sep/Dec) only, taken from the real distinct
months confirmed above, not an assumed or generated date range. The embargo
and test windows are never touched by a fold.


In [20]:
def is_quarter_end(ts: pd.Timestamp) -> bool:
    return ts.month in QUARTER_END_MONTHS

quarter_end_months = [m for m in months if is_quarter_end(m)]
training_quarters = sorted(m for m in quarter_end_months if m <= TRAIN_END)
embargo_quarters = [m for m in quarter_end_months if EMBARGO_START <= m <= EMBARGO_END]
test_quarters = [m for m in quarter_end_months if TEST_START <= m <= TEST_END]

print(f"training quarter-ends: {len(training_quarters)} ({training_quarters[0].date()} to {training_quarters[-1].date()})")
print(f"embargo quarter-ends:  {len(embargo_quarters)} ({[q.date() for q in embargo_quarters]})")
print(f"test quarter-ends:     {len(test_quarters)} ({test_quarters[0].date()} to {test_quarters[-1].date()})")

assert all(q <= TRAIN_END for q in training_quarters)
assert not any(EMBARGO_START <= q <= EMBARGO_END for q in training_quarters)


training quarter-ends: 87 (2000-03-31 to 2021-09-30)
embargo quarter-ends:  1 ([datetime.date(2021, 12, 31)])
test quarter-ends:     17 (2022-03-31 to 2026-03-31)


## 7. Expanding-window CV folds

Fold *k*'s training set grows from the first training quarter through a
boundary; its validation set is the next `CV_FOLD_VAL_QUARTERS` quarters,
never overlapping another fold's. Generated until a full validation window no
longer fits -- never emits a short fold.


In [21]:
def expanding_window_folds(quarters, initial_train_quarters=CV_INITIAL_TRAIN_QUARTERS, val_quarters=CV_FOLD_VAL_QUARTERS):
    quarters = list(quarters)
    folds = []
    train_end_idx = initial_train_quarters
    fold_id = 0
    while train_end_idx + val_quarters <= len(quarters):
        train_slice = quarters[:train_end_idx]
        val_slice = quarters[train_end_idx: train_end_idx + val_quarters]
        folds.append({
            "fold_id": fold_id,
            "train_start": train_slice[0],
            "train_end": train_slice[-1],
            "val_start": val_slice[0],
            "val_end": val_slice[-1],
            "n_train_quarters": len(train_slice),
            "n_val_quarters": len(val_slice),
        })
        fold_id += 1
        train_end_idx += val_quarters
    return folds

folds = expanding_window_folds(training_quarters)
if not folds:
    raise ValueError("STOP AND REPORT: expanding_window_folds produced zero folds")
undersized = [f for f in folds if f["n_val_quarters"] < CV_FOLD_VAL_QUARTERS]
if undersized:
    raise ValueError(f"STOP AND REPORT: undersized validation window(s): {undersized}")

print(f"{len(folds)} folds generated")
pd.DataFrame(folds)


11 folds generated


,fold_id,train_start,train_end,val_start,val_end,n_train_quarters,n_val_quarters
0,0,2000-03-31,2009-12-31,2010-03-31,2010-12-31,40,4
1,1,2000-03-31,2010-12-31,2011-03-31,2011-12-31,44,4
2,2,2000-03-31,2011-12-31,2012-03-31,2012-12-31,48,4
3,3,2000-03-31,2012-12-31,2013-03-31,2013-12-31,52,4
4,4,2000-03-31,2013-12-31,2014-03-31,2014-12-31,56,4
5,5,2000-03-31,2014-12-31,2015-03-31,2015-12-31,60,4
6,6,2000-03-31,2015-12-31,2016-03-31,2016-12-31,64,4
7,7,2000-03-31,2016-12-31,2017-03-31,2017-12-31,68,4
8,8,2000-03-31,2017-12-31,2018-03-31,2018-12-31,72,4
9,9,2000-03-31,2018-12-31,2019-03-31,2019-12-31,76,4


In [22]:
# Training sets strictly grow; validation windows are contiguous, never overlap.
assert all(folds[i]["n_train_quarters"] > folds[i - 1]["n_train_quarters"] for i in range(1, len(folds)))
assert all(folds[i]["val_start"] > folds[i - 1]["val_end"] for i in range(1, len(folds)))
print("first fold:", folds[0])
print("last fold: ", folds[-1])


first fold: {'fold_id': 0, 'train_start': Timestamp('2000-03-31 00:00:00'), 'train_end': Timestamp('2009-12-31 00:00:00'), 'val_start': Timestamp('2010-03-31 00:00:00'), 'val_end': Timestamp('2010-12-31 00:00:00'), 'n_train_quarters': 40, 'n_val_quarters': 4}
last fold:  {'fold_id': 10, 'train_start': Timestamp('2000-03-31 00:00:00'), 'train_end': Timestamp('2019-12-31 00:00:00'), 'val_start': Timestamp('2020-03-31 00:00:00'), 'val_end': Timestamp('2020-12-31 00:00:00'), 'n_train_quarters': 80, 'n_val_quarters': 4}


## 8. Persist folds to JSON; confirm round-trip

Every later modeling notebook should load `../outputs/cv_folds.json` rather
than regenerating folds -- this is what actually guarantees identical CV
splits across every downstream step, instead of silently drifting if the
underlying row/month count changes between runs.


In [23]:
def _serialize_fold(fold):
    out = dict(fold)
    for key in ("train_start", "train_end", "val_start", "val_end"):
        out[key] = out[key].strftime("%Y-%m-%d")
    return out

def save_folds(folds, path=FOLDS_PATH):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    serializable = [_serialize_fold(f) for f in folds]
    with open(path, "w") as fh:
        json.dump(serializable, fh, indent=2)
    return path, serializable

def load_folds(path=FOLDS_PATH):
    with open(path) as fh:
        return json.load(fh)

path, serialized = save_folds(folds)
reloaded = load_folds(path)
print(f"persisted {len(folds)} folds to {path}")
print(f"round-trip exact match (same fold count and boundaries): {reloaded == serialized}")
assert reloaded == serialized


persisted 11 folds to ../outputs/cv_folds.json
round-trip exact match (same fold count and boundaries): True


## 9. Findings

In [24]:
print(f"Training quarter-ends: {len(training_quarters)} ({training_quarters[0].date()} to {training_quarters[-1].date()})")
print(f"Embargo quarter-ends:  {len(embargo_quarters)} ({[q.date() for q in embargo_quarters]})")
print(f"Test quarter-ends:     {len(test_quarters)} ({test_quarters[0].date()} to {test_quarters[-1].date()})")
print(f"CV folds generated: {len(folds)}")

first, last = folds[0], folds[-1]
print(f"Fold 0:  train {first['train_start'].date()}..{first['train_end'].date()} ({first['n_train_quarters']}q) "
      f"-> val {first['val_start'].date()}..{first['val_end'].date()}")
print(f"Fold {len(folds) - 1}: train {last['train_start'].date()}..{last['train_end'].date()} ({last['n_train_quarters']}q) "
      f"-> val {last['val_start'].date()}..{last['val_end'].date()}")


Training quarter-ends: 87 (2000-03-31 to 2021-09-30)
Embargo quarter-ends:  1 ([datetime.date(2021, 12, 31)])
Test quarter-ends:     17 (2022-03-31 to 2026-03-31)
CV folds generated: 11
Fold 0:  train 2000-03-31..2009-12-31 (40q) -> val 2010-03-31..2010-12-31
Fold 10: train 2000-03-31..2019-12-31 (80q) -> val 2020-03-31..2020-12-31


- Training-period quarter-ends, embargo quarter-ends, and test quarter-ends
  (printed above) are all confirmed against real `county_month_check` months,
  not assumed.
- Expanding-window folds (`CV_INITIAL_TRAIN_QUARTERS=40`, `CV_FOLD_VAL_QUARTERS=4`)
  were generated from those real training quarters. Training sets strictly
  grow fold to fold; validation windows are contiguous and never overlap
  (asserted above). Any leftover training-period quarters after the last
  fold's validation window are simply unused by CV -- expected, not an error.
- `../outputs/cv_folds.json` round-trips exactly: reloading it gives back the
  identical fold count and boundaries (asserted above). Every later modeling
  notebook should load this file rather than recomputing a split.
- No "stop and report" condition fired: `TRAIN_END`, `EMBARGO_END`, and
  `TEST_END` all matched real `county_month_check` months, the fold generator
  produced a non-empty set of folds, and every fold had the full
  `CV_FOLD_VAL_QUARTERS` validation window.
- **Not done here:** the minimum-positives-per-fold rule needs a real
  `decline_label`, which doesn't exist in this table yet -- that check
  belongs in whichever later notebook builds the quarterly modeling table
  with real labels, not here.
